<a href="https://colab.research.google.com/github/VictorHugoTesti/FATEC-Processamento_Linguagem_Natural/blob/main/%5BPLN%5D_Aula09_Exercicio0930.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalação e download de recursos

In [18]:
!pip install -q spacy pandas
!python -m spacy download pt_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 77.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import spacy
import pandas as pd

In [20]:
nlp = spacy.load("pt_core_news_sm")

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


# Pipeline 1

In [21]:
corpus_exercicio = [
    "O síndico é muito prestativo e sempre resolve tudo rápido.",
    "A área de lazer não estava limpa quando visitei.",
    "O condomínio é caro, mas a estrutura compensa.",
    "Nunca vi um prédio tão mal administrado.",
]

lexico_positivo = {
    "ótimo", "excelente", "adorar", "maravilhoso", "perfeito", "recomendar",
    "satisfeito", "rápido", "eficiente", "atencioso", "gentil", "confiável",
    "espaçoso", "iluminado", "tranquilo", "seguro", "bom"
}

lexico_negativo = {
    "péssimo", "horrível", "detestar", "decepcionar", "lento", "descaso",
    "problema", "sujo", "escuro", "barulhento", "caro", "insatisfeito",
    "atrasado", "ruim", "complicado", "incomodar"
}

In [22]:
# 1. Classificação de cada comentário
def polaridade_com_negacao(texto, lex_pos, lex_neg):
    doc = nlp(texto)
    positivas = 0
    negativas = 0

    for token in doc:
        lema = token.lemma_.lower()
        negado = any(
            filho.lemma_.lower() in ("não", "nunca", "jamais") and filho.dep_ == "advmod"
            for filho in token.children
        )

        if lema in lex_pos:
            if negado:
                negativas += 1
            else:
                positivas += 1
        elif lema in lex_neg:
            if negado:
                positivas += 1
            else:
                negativas += 1

    if positivas > negativas:
        return "Positivo", positivas, negativas
    elif negativas > positivas:
        return "Negativo", positivas, negativas
    return "Neutro", positivas, negativas

In [23]:
print("--- 1. Classificação de cada comentário (Léxico Original) ---")
for i, texto in enumerate(corpus_exercicio, 1):
    classificacao, pos, neg = polaridade_com_negacao(texto, lexico_positivo, lexico_negativo)
    print(f"Comentário {i}: {classificacao} (pos={pos}, neg={neg}) -> '{texto}'")

--- 1. Classificação de cada comentário (Léxico Original) ---
Comentário 1: Positivo (pos=1, neg=0) -> 'O síndico é muito prestativo e sempre resolve tudo rápido.'
Comentário 2: Neutro (pos=0, neg=0) -> 'A área de lazer não estava limpa quando visitei.'
Comentário 3: Negativo (pos=0, neg=1) -> 'O condomínio é caro, mas a estrutura compensa.'
Comentário 4: Neutro (pos=0, neg=0) -> 'Nunca vi um prédio tão mal administrado.'


In [24]:
# 1. Classificação de cada comentário
print("\n--- 2. Investigação dos resultados inesperados ---")
"""
Resultados esperados vs reais da Etapa 1:
1. "síndico... rápido" -> Foi POSITIVO (pegou 'rápido'), mas ignorou 'prestativo'.
2. "não estava limpa" -> Foi NEUTRO. O algoritmo não encontrou palavras polarizadas.
3. "caro, mas... compensa" -> Foi NEGATIVO (pegou 'caro'). Ignorou 'compensa'.
4. "mal administrado" -> Foi NEUTRO. Ignorou 'mal' e 'administrado'.
"""


--- 2. Investigação dos resultados inesperados ---


'\nResultados esperados vs reais da Etapa 1:\n1. "síndico... rápido" -> Foi POSITIVO (pegou \'rápido\'), mas ignorou \'prestativo\'.\n2. "não estava limpa" -> Foi NEUTRO. O algoritmo não encontrou palavras polarizadas.\n3. "caro, mas... compensa" -> Foi NEGATIVO (pegou \'caro\'). Ignorou \'compensa\'.\n4. "mal administrado" -> Foi NEUTRO. Ignorou \'mal\' e \'administrado\'.\n'

In [25]:
# 3. Ajuste do léxico, se necessário
frases_investigar = [1, 2, 3]
for i in frases_investigar:
    print(f"\nDebug: '{corpus_exercicio[i]}'")
    doc = nlp(corpus_exercicio[i])
    for token in doc:
        print(f"Token: {token.text:12} | Lema: {token.lemma_:12} | Dep: {token.dep_:10} | Head: {token.head.text}")


print("\n--- 3. Ajuste do léxico e Reteste ---")


Debug: 'A área de lazer não estava limpa quando visitei.'
Token: A            | Lema: o            | Dep: det        | Head: área
Token: área         | Lema: área         | Dep: nsubj      | Head: limpa
Token: de           | Lema: de           | Dep: case       | Head: lazer
Token: lazer        | Lema: lazer        | Dep: nmod       | Head: área
Token: não          | Lema: não          | Dep: advmod     | Head: limpa
Token: estava       | Lema: estar        | Dep: cop        | Head: limpa
Token: limpa        | Lema: limpa        | Dep: ROOT       | Head: limpa
Token: quando       | Lema: quando       | Dep: advmod     | Head: visitei
Token: visitei      | Lema: visitar      | Dep: advcl      | Head: limpa
Token: .            | Lema: .            | Dep: punct      | Head: limpa

Debug: 'O condomínio é caro, mas a estrutura compensa.'
Token: O            | Lema: o            | Dep: det        | Head: condomínio
Token: condomínio   | Lema: condomínio   | Dep: nsubj      | Head: caro
Toke

In [26]:
lexico_positivo_novo = lexico_positivo | {"prestativo", "resolver", "limpo", "compensar"}
lexico_negativo_novo = lexico_negativo | {"mal"}

for i, texto in enumerate(corpus_exercicio, 1):
    classificacao, pos, neg = polaridade_com_negacao(texto, lexico_positivo_novo, lexico_negativo_novo)
    print(f"Comentário {i}: {classificacao} (pos={pos}, neg={neg}) -> '{texto}'")

Comentário 1: Positivo (pos=3, neg=0) -> 'O síndico é muito prestativo e sempre resolve tudo rápido.'
Comentário 2: Neutro (pos=0, neg=0) -> 'A área de lazer não estava limpa quando visitei.'
Comentário 3: Negativo (pos=0, neg=1) -> 'O condomínio é caro, mas a estrutura compensa.'
Comentário 4: Negativo (pos=0, neg=1) -> 'Nunca vi um prédio tão mal administrado.'
